In [11]:
import os
import json
from csv import DictReader
from taxonomy import *
from sklearn.metrics import cohen_kappa_score

bugs_directory = "../../results/ESEM_26/bug_reports"
csv_path = "../../results/ESEM_26/classifications.csv"

bugs = {}
classification = {}
llm_classification = []
human_verification = []

with open(csv_path, 'r') as csv_file:
    lines = csv_file.readlines()
    csv_reader = DictReader(lines)
    for row in csv_reader:
        classification[row['id']] = 'TRUE' if 'TRUE' in row['Agreed Classification'] else 'FALSE'
        llm_classification.append(True if 'TRUE' in row['LLM Classification'] else False)
        human_verification.append(True if 'TRUE' in row['Human Verification Result'] else False)

kappa = cohen_kappa_score(llm_classification, human_verification)
print(f"Cohen's Kappa: {kappa}")

for filename in os.listdir(bugs_directory):
    if filename.endswith(".json") and "#" not in filename:
        with open(os.path.join(bugs_directory, filename), 'r') as json_file:
            bug_data = json.load(json_file)
            bug_id = bug_data['id']
            bugs[bug_id] = bug_data
            bug_data['classification'] = classification[bug_id]
            if 'err' in bug_data:
                bug_data['tool'] = 'typechef'
                bug_data['strategy'] = 'family'
                bug_data['type'] = get_warning_type(bug_data)
            elif bug_data['originalAlarm']['alarmType'].isupper():
                bug_data['tool'] = 'infer'
                bug_data['type'] = get_warning_type(bug_data)
            else:
                bug_data['tool'] = 'clang'
                bug_data['type'] = get_warning_type(bug_data)
            if bug_data['strategy'] == 'family':
                line = bug_data['line_number']
                lines = [line, line]
                file = os.path.basename(bug_data['input_file'])
            elif bug_data['strategy'] == 'product':
                line = bug_data['originalAlarm']['line']
                lines = [line, line]
                file = os.path.basename(bug_data['originalAlarm']['fileLocation'])
            else:
                lines = bug_data['lineInputFile']
                file = os.path.basename(bug_data['originalFile'])
            key = (file, tuple(lines), bug_data['type'], bug_data['classification'])
            if 'key' not in bugs:
                bugs['key'] = set()
            bugs['key'].add((bug_id, bug_data['classification']))

filtered_bugs = list(filter(lambda k: "TRUE" in k[1]['classification'], bugs.items()))
print(f"Total bugs: {len(bugs)}")
print(f"Filtered bugs: {len(filtered_bugs)}")

# Group bugs by line number and file
grouped_bugs = {}
venn = {}
for bug_id, bug_data in filtered_bugs:
    if bug_data['strategy'] == 'product':
        line = bug_data['originalAlarm']['line']
        lines = [line, line]
        file = os.path.basename(bug_data['originalAlarm']['fileLocation'])
    elif bug_data['strategy'] == 'transformation':
        lines = bug_data['lineInputFile']
        file = os.path.basename(bug_data['originalFile'])
    else: # family
    key = (file, tuple(lines), bug_data['type'])
    if key not in grouped_bugs:
        grouped_bugs[key] = []
    grouped_bugs[key].append(bug_data)
    if bug_data['strategy'] not in venn:
        venn[bug_data['strategy']] = set()
    venn[bug_data['strategy']].add(key)

# Print the grouped bugs
for key, bugs in grouped_bugs.items():
    print(f"File: {key[0]}, Lines: {key[1]}, Bugs: {len(bugs)}")

# Create the venn diagram
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
venn3(subsets=(venn.get('product', set()), venn.get('transformation', set()), venn.get('family', set())), set_labels=('Product', 'Transformation', 'Family'))
plt.show()

IndentationError: expected an indented block after 'else' statement on line 74 (3134950303.py, line 75)